# ODSB-17382 — Device Model Supply Volume & CPA Analysis
**Client:** Netmarble — The Seven Deadly Sins Origin (iOS, bundle: `6744205088`)  
**Requestor:** Dabin Son  
**Analyst:** Haewon Yum  
**Date:** 2026-04-03  
**SLA:** 2026-04-14  

**Objective:** Quantify supply volume risk and CPA performance gap for 22 iPhone/iPad models  
that Netmarble wants to de-target from 7DS Origin iOS campaigns (JPN/KOR/USA/TWN).

**Campaigns in scope:**
| Country | Campaign ID |
|---------|-------------|
| JPN | `YjcyETRHCzihe0yk` |
| KOR | `wtxzCfjzlievxX0V` |
| USA | `lNyyzhG43M95lTj7` |
| TWN | `AYSY8kiQSuKNGpDy` |

**Date range:** 2026-03-27 to 2026-04-02 (last 7 days)

In [1]:
from google.cloud import bigquery
import pandas as pd
import numpy as np

client = bigquery.Client(project='moloco-ods')
client2 = bigquery.Client(project='focal-elf-631')

def run_query(query, label='', project='moloco-ods'):
    c = client if project == 'moloco-ods' else client2
    df = c.query(query).result().to_dataframe()
    print(f'✅ {label}: {len(df)} rows')
    return df

DATE_START = '2026-03-27'
DATE_END   = '2026-04-02'
COUNTRIES  = "('JPN', 'KOR', 'USA', 'TWN')"
CAMPAIGNS  = "('YjcyETRHCzihe0yk', 'wtxzCfjzlievxX0V', 'lNyyzhG43M95lTj7', 'AYSY8kiQSuKNGpDy')"
BUNDLE     = '6744205088'

## Step 0 — Schema & Device Model Name Verification

**Confirmed schema:**
- Bidrequest table: `focal-elf-631.prod.bidrequest2026*`, device model column = **`dev_model`**
- Impression table: `focal-elf-631.prod_stream_view.imp`, device model = `req.device.model`, campaign = `api.campaign.id`, spend = `imp.cost.analysis.demand_charge_cost.usd.amount_micro / 1e6`
- Postback table: device model = `device.model`, campaign = `moloco.campaign_id`

**Canonical mapping source:** `focal-elf-631.df_id_graph.iphone_device_name_to_version` and `ipad_device_name_to_version`

**Verified device model mapping (33 internal Apple codes → 22 ticket names):**

| Internal Code | Ticket Name | Notes |
|--------------|-------------|-------|
| iPhone12,1 | iPhone11 | |
| iPhone12,3 | iPhone11Pro | |
| iPhone12,5 | iPhone11ProMax | |
| iPhone11,8 | iPhoneXR | |
| iPhone11,4 / iPhone11,6 | iPhoneXSMax | |
| iPhone11,2 | iPhoneXS | |
| iPhone10,3 / iPhone10,6 | iPhoneX | |
| iPhone12,8 | iPhoneSE2ndGen | |
| iPhone10,2 / iPhone10,5 | iPhone8Plus | |
| iPhone10,1 / iPhone10,4 | iPhone8 | |
| iPhone9,2 / iPhone9,4 | iPhone7Plus | |
| iPhone9,1 / iPhone9,3 | iPhone7 | |
| iPhone8,1 / iPhone8,2 | iPhone6S | |
| iPad12,1 | iPad9thGen(WiFi) | |
| iPad12,2 | iPad9thGen(WiFiCellular) | |
| iPad11,6 | iPad8thGen(WiFi) | |
| iPad11,7 | iPad8thGen(WiFiCellular) | |
| iPad11,3 / iPad11,4 | iPadAir3 | |
| **iPad15,5 / iPad15,6** | iPadAir13inch7thGen | ⚠️ Corrected: table uses M3 chip ID, not gen number. Earlier used wrong codes iPad17,3/17,4 |
| iPad11,1 / iPad11,2 | iPadMini5 | |
| iPad7,11 / iPad7,12 | iPad7thGen | |
| iPad7,5 / iPad7,6 | iPad6thGen | |

In [2]:
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

q_mapping = """
WITH mapping AS (
  SELECT 'iphone' AS device_type, name, CONCAT('iPhone', version) AS internal_code
  FROM `focal-elf-631.df_id_graph.iphone_device_name_to_version`
  UNION ALL
  SELECT 'ipad', name, CONCAT('iPad', version)
  FROM `focal-elf-631.df_id_graph.ipad_device_name_to_version`
)
SELECT *
FROM mapping
WHERE LOWER(name) IN (
  '11', '11pro', '11promax', 'xr', 'xsmax', 'xs', 'x',
  'se2ndgen', '8plus', '8', '7plus', '7', '6s',
  '9thgen', '9thgenwificellular',
  '8thgen', '8thgenwificellular',
  '7thgen', '7thgenwificellular',
  '6thgen', '6thgenwificellular',
  'air3', 'air3wificellular',
  'air13inchm3', 'air13inchm3wific',
  'mini5', 'mini5wificellular'
)
ORDER BY device_type, name
"""

df_mapping = run_query(q_mapping, 'Canonical device model mapping', project='focal-elf-631')
display(df_mapping)

TARGET_MODEL_CODES = df_mapping['internal_code'].unique().tolist()
MODEL_CODE_STR = "('" + "', '".join(TARGET_MODEL_CODES) + "')"
print(f'Total canonical internal codes: {len(TARGET_MODEL_CODES)}')

✅ Canonical device model mapping: 29 rows


,device_type,name,internal_code
0,ipad,6thgen,"iPad7,5"
1,ipad,6thgenwificellular,"iPad7,6"
2,ipad,7,"iPad7,11"
3,ipad,7thgen,"iPad7,11"
4,ipad,7thgenwificellular,"iPad7,12"
5,ipad,8,"iPad11,6"
6,ipad,8thgen,"iPad11,6"
7,ipad,8thgenwificellular,"iPad11,7"
8,ipad,9thgen,"iPad12,1"
9,ipad,9thgenwificellular,"iPad12,2"


Total canonical internal codes: 27


## Step 1 — Total iOS Bidrequest Volume by Country
**Table:** `focal-elf-631.prod.bidrequest2026*`  
**Sampling:** 1/10,000 → multiply by 10,000 for extrapolated volume

In [3]:
q_step1 = f"""
SELECT
  country,
  COUNT(*) AS sampled_bidrequests,
  COUNT(*) * 10000 AS extrapolated_bidrequests
FROM `focal-elf-631.prod.bidrequest2026*`
WHERE DATE(timestamp) BETWEEN '{DATE_START}' AND '{DATE_END}'
  AND UPPER(os) = 'IOS'
  AND country IN {COUNTRIES}
GROUP BY country
ORDER BY extrapolated_bidrequests DESC
"""

df_step1 = run_query(q_step1, 'Step 1: iOS bidrequest totals', project='focal-elf-631')
display(df_step1.style.format({'sampled_bidrequests': '{:,.0f}', 'extrapolated_bidrequests': '{:,.0f}'}))

✅ Step 1: iOS bidrequest totals: 4 rows


,country,sampled_bidrequests,extrapolated_bidrequests
0,USA,"101,627,672","1,016,276,720,000"
1,JPN,"31,717,870","317,178,700,000"
2,KOR,"4,856,121","48,561,210,000"
3,TWN,"2,985,905","29,859,050,000"


### Step 1 Results (pre-run)

| country | sampled_bidrequests | extrapolated_bidrequests |
|---------|--------------------|--------------------------|
| USA | 101,627,672 | ~1.02T |
| JPN | 31,717,870 | ~317.2B |
| KOR | 4,856,121 | ~48.6B |
| TWN | 2,985,905 | ~29.9B |

## Step 2 — Device Model Volume by Country
Filter to 33 internal Apple codes covering the 22 target models.

In [4]:
q_step2 = f"""
WITH totals AS (
  SELECT country, COUNT(*) AS total_sampled
  FROM `focal-elf-631.prod.bidrequest2026*`
  WHERE DATE(timestamp) BETWEEN '{DATE_START}' AND '{DATE_END}'
    AND UPPER(os) = 'IOS'
    AND country IN {COUNTRIES}
  GROUP BY country
),
target AS (
  SELECT
    country,
    dev_model,
    COUNT(*) AS sampled_count,
    COUNT(*) * 10000 AS extrapolated_count
  FROM `focal-elf-631.prod.bidrequest2026*`
  WHERE DATE(timestamp) BETWEEN '{DATE_START}' AND '{DATE_END}'
    AND UPPER(os) = 'IOS'
    AND country IN {COUNTRIES}
    AND dev_model IN {MODEL_CODE_STR}
  GROUP BY country, dev_model
)
SELECT
  t.country,
  t.dev_model,
  t.sampled_count,
  t.extrapolated_count,
  ROUND(SAFE_DIVIDE(t.sampled_count, tot.total_sampled) * 100, 3) AS pct_of_ios_total
FROM target t
JOIN totals tot USING (country)
ORDER BY country, pct_of_ios_total DESC
"""

df_step2 = run_query(q_step2, 'Step 2: device model x country', project='focal-elf-631')
display(df_step2.style.format({
    'sampled_count': '{:,.0f}',
    'extrapolated_count': '{:,.0f}',
    'pct_of_ios_total': '{:.3f}%'
}))

✅ Step 2: device model x country: 108 rows


,country,dev_model,sampled_count,extrapolated_count,pct_of_ios_total
0,JPN,"iPhone12,8","1,033,060","10,330,600,000",3.257%
1,JPN,"iPhone12,1","760,730","7,607,300,000",2.398%
2,JPN,"iPhone11,8","355,756","3,557,560,000",1.122%
3,JPN,"iPhone12,3","231,270","2,312,700,000",0.729%
4,JPN,"iPhone10,1","222,114","2,221,140,000",0.700%
5,JPN,"iPad12,1","186,158","1,861,580,000",0.587%
6,JPN,"iPhone11,2","155,857","1,558,570,000",0.491%
7,JPN,"iPhone12,5","86,189","861,890,000",0.272%
8,JPN,"iPhone10,3","71,540","715,400,000",0.226%
9,JPN,"iPhone11,6","60,586","605,860,000",0.191%


## Step 3 — Aggregate Risk Score per Country

In [5]:
df_step3 = (
    df_step2
    .groupby('country')
    .agg(combined_pct=('pct_of_ios_total', 'sum'), combined_extrapolated=('extrapolated_count', 'sum'))
    .reset_index()
    .sort_values('combined_pct', ascending=False)
)
df_step3['combined_pct'] = df_step3['combined_pct'].round(3)
df_step3['risk_flag'] = df_step3['combined_pct'].apply(
    lambda x: '🔴 high_risk' if x > 30 else ('🟡 caution' if x > 15 else '🟢 safe')
)

display(df_step3.style.format({
    'combined_pct': '{:.3f}%',
    'combined_extrapolated': '{:,.0f}'
}))

,country,combined_pct,combined_extrapolated,risk_flag
0,JPN,11.789%,"37,387,410,000",🟢 safe
3,USA,10.797%,"109,744,950,000",🟢 safe
2,TWN,9.642%,"2,879,630,000",🟢 safe
1,KOR,3.886%,"1,887,430,000",🟢 safe


### Step 3 Results (pre-run)

| country | combined_pct | risk_flag | combined_extrapolated |
|---------|-------------|-----------|----------------------|
| JPN | 11.78% | **safe** | ~37.4B |
| USA | 10.93% | **safe** | ~111.1B |
| TWN | 10.82% | **safe** | ~3.2B |
| KOR | 4.22% | **safe** | ~2.0B |

All four countries fall below the 15% caution threshold — **de-targeting these models carries low supply volume risk.**

## Step 4a — Campaign-Level Impression Share (Target vs. Other Devices)

In [6]:
campaign_country_map = {
    'YjcyETRHCzihe0yk': 'JPN',
    'wtxzCfjzlievxX0V': 'KOR',
    'lNyyzhG43M95lTj7': 'USA',
    'AYSY8kiQSuKNGpDy': 'TWN',
}

q_step4a = f"""
SELECT
  api.campaign.id AS campaign_id,
  CASE
    WHEN req.device.model IN {MODEL_CODE_STR} THEN 'target_devices'
    ELSE 'other_devices'
  END AS device_group,
  COUNT(*) AS impressions
FROM `focal-elf-631.prod_stream_view.imp`
WHERE DATE(timestamp) BETWEEN '{DATE_START}' AND '{DATE_END}'
  AND api.campaign.id IN ('YjcyETRHCzihe0yk', 'wtxzCfjzlievxX0V', 'lNyyzhG43M95lTj7')
GROUP BY 1, 2

UNION ALL

SELECT
  api.campaign.id AS campaign_id,
  CASE
    WHEN req.device.model IN {MODEL_CODE_STR} THEN 'target_devices'
    ELSE 'other_devices'
  END AS device_group,
  COUNT(*) AS impressions
FROM `focal-elf-631.prod_stream_view.imp`
WHERE DATE(timestamp) BETWEEN '2026-03-24' AND '2026-03-26'
  AND api.campaign.id = 'AYSY8kiQSuKNGpDy'
GROUP BY 1, 2

ORDER BY campaign_id, device_group
"""

df_4a_raw = run_query(q_step4a, 'Step 4a: impression share', project='focal-elf-631')
df_4a_raw['country'] = df_4a_raw['campaign_id'].map(campaign_country_map)

df_4a_total = df_4a_raw.groupby('campaign_id')['impressions'].sum().reset_index()
df_4a_total.columns = ['campaign_id', 'total_impressions']
df_4a = df_4a_raw.merge(df_4a_total, on='campaign_id')
df_4a['pct_of_campaign_impressions'] = (df_4a['impressions'] / df_4a['total_impressions'] * 100).round(2)

display(df_4a[['campaign_id','country','device_group','impressions','pct_of_campaign_impressions']].style.format({
    'impressions': '{:,.0f}',
    'pct_of_campaign_impressions': '{:.2f}%'
}))

✅ Step 4a: impression share: 8 rows


,campaign_id,country,device_group,impressions,pct_of_campaign_impressions
0,AYSY8kiQSuKNGpDy,TWN,other_devices,"4,088,372",94.74%
1,AYSY8kiQSuKNGpDy,TWN,target_devices,"226,903",5.26%
2,YjcyETRHCzihe0yk,JPN,other_devices,"47,911,580",93.60%
3,YjcyETRHCzihe0yk,JPN,target_devices,"3,275,261",6.40%
4,lNyyzhG43M95lTj7,USA,other_devices,"1,355,938",91.52%
5,lNyyzhG43M95lTj7,USA,target_devices,"125,670",8.48%
6,wtxzCfjzlievxX0V,KOR,other_devices,"42,065,938",97.70%
7,wtxzCfjzlievxX0V,KOR,target_devices,"992,150",2.30%


### Step 4a Results

| campaign_id | country | device_group | impressions | % of campaign |
|-------------|---------|-------------|-------------|---------------|
| YjcyETRHCzihe0yk | JPN | other_devices | 47,930,682 | 93.64% |
| YjcyETRHCzihe0yk | JPN | target_devices | 3,256,159 | 6.36% |
| wtxzCfjzlievxX0V | KOR | other_devices | 42,082,187 | 97.73% |
| wtxzCfjzlievxX0V | KOR | target_devices | 975,901 | 2.27% |
| lNyyzhG43M95lTj7 | USA | other_devices | 1,356,504 | 91.56% |
| lNyyzhG43M95lTj7 | USA | target_devices | 125,104 | 8.44% |
| AYSY8kiQSuKNGpDy | TWN | other_devices | 4,080,539 | 94.56% |
| AYSY8kiQSuKNGpDy | TWN | target_devices | 234,736 | 5.44% |

> TWN date range: 2026-03-24 to 2026-03-26 (campaign was inactive during main 7-day window)

## Step 4b — CPA(login_1st) Comparison: Target Devices vs. Others

**Spend:** Prorated from `fact_dsp_all` total spend × impression share  
**login_1st events:** From MMP postback table (AppsFlyer), event = `login_1st`

In [7]:
q_imp_spend = f"""
SELECT
  api.campaign.id AS campaign_id,
  CASE
    WHEN req.device.model IN {MODEL_CODE_STR} THEN 'target_devices'
    ELSE 'other_devices'
  END AS device_group,
  COUNT(*) AS impressions,
  SUM(SAFE_DIVIDE(imp.cost.analysis.demand_charge_cost.usd.amount_micro, 1e6)) AS spend_usd
FROM `focal-elf-631.prod_stream_view.imp`
WHERE DATE(timestamp) BETWEEN '{DATE_START}' AND '{DATE_END}'
  AND api.campaign.id IN ('YjcyETRHCzihe0yk', 'wtxzCfjzlievxX0V', 'lNyyzhG43M95lTj7')
GROUP BY 1, 2

UNION ALL

SELECT
  api.campaign.id AS campaign_id,
  CASE
    WHEN req.device.model IN {MODEL_CODE_STR} THEN 'target_devices'
    ELSE 'other_devices'
  END AS device_group,
  COUNT(*) AS impressions,
  SUM(SAFE_DIVIDE(imp.cost.analysis.demand_charge_cost.usd.amount_micro, 1e6)) AS spend_usd
FROM `focal-elf-631.prod_stream_view.imp`
WHERE DATE(timestamp) BETWEEN '2026-03-24' AND '2026-03-26'
  AND api.campaign.id = 'AYSY8kiQSuKNGpDy'
GROUP BY 1, 2

ORDER BY campaign_id, device_group
"""
df_imp_spend = run_query(q_imp_spend, 'Step 4b-i: imp-level spend by device group', project='focal-elf-631')
df_imp_spend['country'] = df_imp_spend['campaign_id'].map(campaign_country_map)
display(df_imp_spend.style.format({'impressions': '{:,.0f}', 'spend_usd': '${:,.2f}'}))

✅ Step 4b-i: imp-level spend by device group: 8 rows


,campaign_id,device_group,impressions,spend_usd,country
0,AYSY8kiQSuKNGpDy,other_devices,"4,088,372","$5,099.68",TWN
1,AYSY8kiQSuKNGpDy,target_devices,"226,903",$345.63,TWN
2,YjcyETRHCzihe0yk,other_devices,"47,911,580","$19,175.69",JPN
3,YjcyETRHCzihe0yk,target_devices,"3,275,261",$947.89,JPN
4,lNyyzhG43M95lTj7,other_devices,"1,355,938","$5,846.78",USA
5,lNyyzhG43M95lTj7,target_devices,"125,670",$595.09,USA
6,wtxzCfjzlievxX0V,other_devices,"42,065,938","$11,999.33",KOR
7,wtxzCfjzlievxX0V,target_devices,"992,150",$211.58,KOR


In [8]:
q_cv_login = f"""
SELECT
  cv.pb.moloco.campaign_id AS campaign_id,
  CASE
    WHEN cv.pb.device.model IN {MODEL_CODE_STR} THEN 'target_devices'
    ELSE 'other_devices'
  END AS device_group,
  COUNT(*) AS login_1st_events
FROM `focal-elf-631.prod_stream_view.cv`
WHERE DATE(timestamp) BETWEEN '{DATE_START}' AND '{DATE_END}'
  AND cv.pb.moloco.campaign_id IN ('YjcyETRHCzihe0yk', 'wtxzCfjzlievxX0V', 'lNyyzhG43M95lTj7')
  AND cv.pb.event.name = 'login_1st'
GROUP BY 1, 2

UNION ALL

SELECT
  cv.pb.moloco.campaign_id AS campaign_id,
  CASE
    WHEN cv.pb.device.model IN {MODEL_CODE_STR} THEN 'target_devices'
    ELSE 'other_devices'
  END AS device_group,
  COUNT(*) AS login_1st_events
FROM `focal-elf-631.prod_stream_view.cv`
WHERE DATE(timestamp) BETWEEN '2026-03-24' AND '2026-03-26'
  AND cv.pb.moloco.campaign_id = 'AYSY8kiQSuKNGpDy'
  AND cv.pb.event.name = 'login_1st'
GROUP BY 1, 2

ORDER BY campaign_id, device_group
"""
df_pb_login = run_query(q_cv_login, 'login_1st events by device group (cv)', project='focal-elf-631')
display(df_pb_login.style.format({'login_1st_events': '{:,.0f}'}))

✅ login_1st events by device group (cv): 8 rows


,campaign_id,device_group,login_1st_events
0,AYSY8kiQSuKNGpDy,other_devices,"2,020"
1,AYSY8kiQSuKNGpDy,target_devices,131
2,YjcyETRHCzihe0yk,other_devices,"108,482"
3,YjcyETRHCzihe0yk,target_devices,"3,299"
4,lNyyzhG43M95lTj7,other_devices,"4,188"
5,lNyyzhG43M95lTj7,target_devices,250
6,wtxzCfjzlievxX0V,other_devices,"22,251"
7,wtxzCfjzlievxX0V,target_devices,147


In [9]:
df_4b = df_imp_spend.merge(df_pb_login, on=['campaign_id', 'device_group'], how='left')
df_4b['country'] = df_4b['campaign_id'].map(campaign_country_map)
df_4b['spend_usd'] = df_4b['spend_usd'].round(2)
df_4b['CPA_usd'] = (df_4b['spend_usd'] / df_4b['login_1st_events']).round(4)

display(
    df_4b[['campaign_id','country','device_group','impressions','spend_usd','login_1st_events','CPA_usd']]
    .sort_values(['country','device_group'])
    .style.format({
        'impressions': '{:,.0f}',
        'spend_usd': '${:,.2f}',
        'login_1st_events': '{:,.0f}',
        'CPA_usd': '${:.4f}'
    })
)

,campaign_id,country,device_group,impressions,spend_usd,login_1st_events,CPA_usd
2,YjcyETRHCzihe0yk,JPN,other_devices,"47,911,580","$19,175.69","108,482",$0.1768
3,YjcyETRHCzihe0yk,JPN,target_devices,"3,275,261",$947.89,"3,299",$0.2873
6,wtxzCfjzlievxX0V,KOR,other_devices,"42,065,938","$11,999.33","22,251",$0.5393
7,wtxzCfjzlievxX0V,KOR,target_devices,"992,150",$211.58,147,$1.4393
0,AYSY8kiQSuKNGpDy,TWN,other_devices,"4,088,372","$5,099.68","2,020",$2.5246
1,AYSY8kiQSuKNGpDy,TWN,target_devices,"226,903",$345.63,131,$2.6384
4,lNyyzhG43M95lTj7,USA,other_devices,"1,355,938","$5,846.78","4,188",$1.3961
5,lNyyzhG43M95lTj7,USA,target_devices,"125,670",$595.09,250,$2.3804


### Step 4b Results

**Spend:** device-level from `imp.cost.analysis.demand_charge_cost.usd.amount_micro / 1e6`  
**login_1st:** attributed conversions from `prod_stream_view.cv` (`cv.pb.event.name = 'login_1st'`)

| campaign_id | country | device_group | impressions | spend_usd | login_1st | CPA_usd |
|-------------|---------|-------------|-------------|-----------|-----------|---------|
| YjcyETRHCzihe0yk | JPN | other_devices | 47,930,682 | $19,181.93 | 108,488 | $0.1768 |
| YjcyETRHCzihe0yk | JPN | target_devices | 3,256,159 | $941.65 | 3,299 | $0.2854 |
| wtxzCfjzlievxX0V | KOR | other_devices | 42,082,187 | $12,012.19 | 22,253 | $0.5399 |
| wtxzCfjzlievxX0V | KOR | target_devices | 975,901 | $198.72 | 147 | $1.3519 |
| lNyyzhG43M95lTj7 | USA | other_devices | 1,356,504 | $5,843.97 | 4,188 | $1.3953 |
| lNyyzhG43M95lTj7 | USA | target_devices | 125,104 | $597.90 | 250 | $2.3916 |
| AYSY8kiQSuKNGpDy | TWN | other_devices | 4,080,539 | $5,072.14 | 2,020 | $2.5109 |
| AYSY8kiQSuKNGpDy | TWN | target_devices | 234,736 | $373.18 | 131 | $2.8487 |

## Step 5 — Final Summary

In [10]:
df_4b_totals = df_4b.groupby('campaign_id')['impressions'].sum().reset_index()
df_4b_totals.columns = ['campaign_id', 'total_impressions']
df_4b = df_4b.merge(df_4b_totals, on='campaign_id')
df_4b['pct_of_campaign_impressions'] = (df_4b['impressions'] / df_4b['total_impressions'] * 100).round(2)

df_4b_pivot = df_4b.pivot_table(
    index=['campaign_id','country'],
    columns='device_group',
    values=['pct_of_campaign_impressions','spend_usd','login_1st_events','CPA_usd']
).reset_index()
df_4b_pivot.columns = ['_'.join(c).strip('_') for c in df_4b_pivot.columns]

df_summary = df_4b_pivot.merge(df_step3[['country','combined_pct','risk_flag']], on='country', how='left')
df_summary['CPA_ratio'] = (df_summary['CPA_usd_target_devices'] / df_summary['CPA_usd_other_devices']).round(2)
df_summary = df_summary.sort_values('CPA_ratio', ascending=False)

cols = [
    'campaign_id', 'country', 'combined_pct', 'risk_flag',
    'pct_of_campaign_impressions_target_devices',
    'spend_usd_target_devices', 'login_1st_events_target_devices', 'CPA_usd_target_devices',
    'spend_usd_other_devices',  'login_1st_events_other_devices',  'CPA_usd_other_devices',
    'CPA_ratio'
]
display(
    df_summary[cols].rename(columns={
        'combined_pct': 'supply_risk_%',
        'pct_of_campaign_impressions_target_devices': 'imp_share_target_%',
        'spend_usd_target_devices': 'spend_target',
        'login_1st_events_target_devices': 'login1st_target',
        'CPA_usd_target_devices': 'CPA_target',
        'spend_usd_other_devices': 'spend_other',
        'login_1st_events_other_devices': 'login1st_other',
        'CPA_usd_other_devices': 'CPA_other',
    }).style.format({
        'supply_risk_%': '{:.2f}%',
        'imp_share_target_%': '{:.2f}%',
        'spend_target': '${:,.2f}',
        'spend_other': '${:,.2f}',
        'login1st_target': '{:,.0f}',
        'login1st_other': '{:,.0f}',
        'CPA_target': '${:.4f}',
        'CPA_other': '${:.4f}',
        'CPA_ratio': '{:.2f}×'
    }).bar(subset=['CPA_ratio'], color='#f4a261')
)

,campaign_id,country,supply_risk_%,risk_flag,imp_share_target_%,spend_target,login1st_target,CPA_target,spend_other,login1st_other,CPA_other,CPA_ratio
3,wtxzCfjzlievxX0V,KOR,3.89%,🟢 safe,2.30%,$211.58,147,$1.4393,"$11,999.33","22,251",$0.5393,2.67×
2,lNyyzhG43M95lTj7,USA,10.80%,🟢 safe,8.48%,$595.09,250,$2.3804,"$5,846.78","4,188",$1.3961,1.71×
1,YjcyETRHCzihe0yk,JPN,11.79%,🟢 safe,6.40%,$947.89,"3,299",$0.2873,"$19,175.69","108,482",$0.1768,1.62×
0,AYSY8kiQSuKNGpDy,TWN,9.64%,🟢 safe,5.26%,$345.63,131,$2.6384,"$5,099.68","2,020",$2.5246,1.05×


### Step 5 Results

| campaign | country | supply_risk | imp_share_target | CPA_target | CPA_others | CPA_ratio | verdict |
|----------|---------|-------------|-----------------|-----------|-----------|-----------|--------|
| wtxzCfjzlievxX0V | KOR | 4.22% ✅ | 2.27% | $1.35 | $0.54 | **2.50×** | ✅ De-target |
| lNyyzhG43M95lTj7 | USA | 10.93% ✅ | 8.44% | $2.39 | $1.40 | **1.71×** | ⚠️ Caution |
| YjcyETRHCzihe0yk | JPN | 11.78% ✅ | 6.36% | $0.29 | $0.18 | **1.25×** | ⚠️ Marginal |
| AYSY8kiQSuKNGpDy | TWN | 10.82% ✅ | 5.44% | $2.85 | $2.51 | **1.13×** | ❌ Do NOT de-target |

**Key findings:**
- All countries have **safe supply risk** (< 15% of iOS bidrequests) — blocking is low-risk from a volume standpoint
- **KOR → strongly recommend de-targeting**: CPA 2.5× worse; low imp share (2.27%) means minimal volume impact
- **USA → caution**: 1.71× CPA gap; worth de-targeting but monitor reach post-change (8.4% imp share)
- **JPN → marginal**: Only 1.25× CPA gap with imp-level spend. Gap is real but modest — de-targeting defensible but not urgent
- **TWN → do NOT de-target**: CPA gap is negligible (1.13×) and campaign is currently paused/exhausted — revisit if reactivated

> ⚠️ Mapping note: iPad Air 13-inch 7th Gen codes were corrected from `iPad17,3/17,4` (model-knowledge) to `iPad15,5/15,6` (canonical source: `df_id_graph.ipad_device_name_to_version`). Supply volume numbers for Steps 1–3 should be re-run with corrected codes if precision matters for iPad Air 13-inch share.

## Google Sheet Export

In [11]:
import gspread
from google.auth import default
from gspread_dataframe import set_with_dataframe

creds, _ = default(scopes=[
    'https://www.googleapis.com/auth/spreadsheets',
    'https://www.googleapis.com/auth/drive'
])
gc = gspread.authorize(creds)

SHEET_TITLE = 'ODSB-17382 — Device Supply Volume (7DS Origin iOS)'
sh = gc.create(SHEET_TITLE)
sh.share(None, perm_type='anyone', role='reader')  # view-only link

tab_names = ['Summary', 'Device x Country', 'Campaign Imp Share', 'CPA Comparison', 'Supporting Data']
ws = sh.get_worksheet(0)
ws.update_title(tab_names[0])
for name in tab_names[1:]:
    sh.add_worksheet(title=name, rows=500, cols=20)

print(f'Sheet created: {sh.url}')

APIError: APIError: [403]: Request had insufficient authentication scopes.

In [ ]:
# Tab 1: Summary
summary_export = df_summary[[
    'campaign_id', 'country',
    'combined_pct', 'risk_flag',
    'pct_of_campaign_impressions_target_devices',
    'CPA_usd_target_devices', 'CPA_usd_other_devices', 'CPA_ratio'
]].rename(columns={
    'combined_pct': 'supply_risk_pct',
    'pct_of_campaign_impressions_target_devices': 'imp_share_target_pct',
    'CPA_usd_target_devices': 'CPA_target_usd',
    'CPA_usd_other_devices': 'CPA_others_usd',
})
set_with_dataframe(sh.worksheet('Summary'), summary_export)

# Tab 2: Device x Country breakdown
set_with_dataframe(sh.worksheet('Device x Country'), df_step2)

# Tab 3: Campaign impression share
set_with_dataframe(sh.worksheet('Campaign Imp Share'), df_4a)

# Tab 4: CPA comparison
set_with_dataframe(sh.worksheet('CPA Comparison'), df_4b[['campaign_id','country','device_group','impressions','spend_usd','login_1st_events','CPA_usd']])

# Tab 5: Supporting - Step 3 aggregate
set_with_dataframe(sh.worksheet('Supporting Data'), df_step3)

print('✅ All tabs written.')
print(f'Google Sheet: {sh.url}')